# Monte Carlo First Trick Strategy Analysis - Medium Hand

This notebook compares two first-trick strategies when you're first to play after the dealer with a medium hand:

- **Strategy 1**: Lead with low trump (9) as a "sacrificial lamb" to draw out other trump, then use Queen trump to win later tricks
- **Strategy 2**: Lead with high off-trump (Ace) hoping everyone follows suit and you win

## Hand Requirements
- Exactly 2 trump cards: 9 and Queen of trump suit
- Ace of a non-trump suit
- High card (10/J/Q/K) of a different non-trump suit
- Hand strength in medium range (100-180)


In [ ]:
import sys
from pathlib import Path
import random
import statistics
from typing import List, Optional, Tuple
import multiprocessing as mp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eucher.cards import Card, Deck, Rank, Suit
from eucher.game import Game
from eucher.players.computer.heuristic import HeuristicPlayer
from eucher.players.profiles import AIPlayer, PlayerProfile
from eucher.rules import RulesEngine
from eucher.trump import TrumpSelector

print("Imports successful")


## Helper Classes and Functions


In [ ]:
class ForcedFirstTrickProfile(PlayerProfile):
    """Player profile that forces a specific card on the first trick."""

    def __init__(self, forced_card: Optional[Card], base_profile: PlayerProfile) -> None:
        """
        Initialize the forced first trick profile.

        Parameters
        ----------
        forced_card : Optional[Card]
            The card to play on the first trick. If None, uses base profile strategy.
        base_profile : PlayerProfile
            The base profile to use for all other decisions.
        """
        self.forced_card: Optional[Card] = forced_card
        self.base_profile: PlayerProfile = base_profile
        self.rules = RulesEngine()

    def decide_order_up(
        self, player: "Player", turned_card: Card, dealer_id: int, trump_suit: Optional[Suit]
    ) -> bool:
        """Delegate to base profile."""
        return self.base_profile.decide_order_up(player, turned_card, dealer_id, trump_suit)

    def decide_call_trump(
        self,
        player: "Player",
        turned_card: Card,
        trump_suit: Optional[Suit],
        must_choose: bool = False,
    ) -> Optional[Suit]:
        """Delegate to base profile."""
        return self.base_profile.decide_call_trump(player, turned_card, trump_suit, must_choose)

    def choose_card_to_discard(
        self, player: "Player", turned_card: Optional[Card] = None, ordered_up_by: Optional[str] = None
    ) -> Card:
        """Delegate to base profile."""
        return self.base_profile.choose_card_to_discard(player, turned_card, ordered_up_by)

    def play_card(
        self,
        player: "Player",
        led_suit: Optional[Suit],
        trump_suit: Optional[Suit],
        trick_cards: List[Card],
        trick_player_ids: List[int],
    ) -> Card:
        """
        Play card, forcing the specified card on the first trick.

        Parameters
        ----------
        player : Player
            The player making the decision.
        led_suit : Optional[Suit]
            The suit that was led, if any.
        trump_suit : Optional[Suit]
            The current trump suit, if any.
        trick_cards : List[Card]
            Cards already played in the trick.
        trick_player_ids : List[int]
            Player IDs who played each card.

        Returns
        -------
        Card
            The card to play.
        """
        # Check if this is the first trick (no led suit, no cards played)
        is_first_trick = led_suit is None and len(trick_cards) == 0

        if is_first_trick and self.forced_card is not None and self.forced_card in player.hand:
            # Validate that the forced card is a legal play
            valid_cards = self.rules.get_valid_plays(player.hand, led_suit, trump_suit)
            if self.forced_card in valid_cards:
                return self.forced_card

        # Use base profile for all other cases
        return self.base_profile.play_card(player, led_suit, trump_suit, trick_cards, trick_player_ids)


In [ ]:
def is_card_trump(card: Card, trump_suit: Suit) -> bool:
    """
    Check if a card is a trump card.

    Parameters
    ----------
    card : Card
        The card to check.
    trump_suit : Suit
        The current trump suit.

    Returns
    -------
    bool
        True if card is trump, False otherwise.
    """
    # Right Bower (Jack of trump suit)
    if card.rank == Rank.JACK and card.suit == trump_suit:
        return True
    # Left Bower (Jack of same color as trump)
    if card.rank == Rank.JACK:
        trump_card = Card(trump_suit, Rank.ACE)  # Dummy card for color check
        if card.is_same_color(trump_card):
            return True
    # Regular trump suit card
    return card.suit == trump_suit


def find_qualifying_hands(
    num_hands: int = 100,
    min_strength: float = 100.0,
    max_strength: float = 180.0,
    seed_base: int = 42,
) -> List[Tuple[List[Card], Card, Suit]]:
    """
    Generate hands that meet the criteria for this analysis.

    Parameters
    ----------
    num_hands : int
        Number of qualifying hands to find.
    min_strength : float
        Minimum hand strength.
    max_strength : float
        Maximum hand strength.
    seed_base : int
        Base seed for random number generation.

    Returns
    -------
    List[Tuple[List[Card], Card, Suit]]
        List of (hand, turned_card, trump_suit) tuples.
    """
    player = HeuristicPlayer()
    qualifying_hands = []
    seed = seed_base

    print(f"Searching for {num_hands} qualifying hands...")
    print(f"Hand strength range: {min_strength}-{max_strength}")

    attempts = 0
    max_attempts = num_hands * 1000  # Prevent infinite loops

    while len(qualifying_hands) < num_hands and attempts < max_attempts:
        attempts += 1
        random.seed(seed)
        np.random.seed(seed)

        # Generate a random hand
        deck = Deck()
        deck.shuffle()
        hand = deck.deal(5)
        turned_card = deck.draw_one()

        # Try each suit as trump
        for trump_suit in Suit:
            # Check if hand has required cards
            trump_cards = [c for c in hand if is_card_trump(c, trump_suit)]
            off_trump_cards = [c for c in hand if not is_card_trump(c, trump_suit)]

            # Must have exactly 2 trump: 9 and Queen
            trump_nines = [c for c in trump_cards if c.rank == Rank.NINE]
            trump_queens = [c for c in trump_cards if c.rank == Rank.QUEEN]

            if len(trump_nines) != 1 or len(trump_queens) != 1:
                continue

            # Must have Ace off-trump
            off_trump_aces = [c for c in off_trump_cards if c.rank == Rank.ACE]
            if len(off_trump_aces) != 1:
                continue

            # Must have high card (10/J/Q/K) of different off-trump suit
            ace_suit = off_trump_aces[0].suit
            high_off_trump = [
                c
                for c in off_trump_cards
                if c.suit != ace_suit
                and c.rank in (Rank.TEN, Rank.JACK, Rank.QUEEN, Rank.KING)
            ]
            if len(high_off_trump) == 0:
                continue

            # Check hand strength
            strength = player._evaluate_hand_strength(hand, trump_suit)
            if min_strength <= strength <= max_strength:
                qualifying_hands.append((hand.copy(), turned_card, trump_suit))
                if len(qualifying_hands) % 10 == 0:
                    print(f"  Found {len(qualifying_hands)} qualifying hands...")
                break

        seed += 1

    print(f"\nFound {len(qualifying_hands)} qualifying hands after {attempts} attempts")
    return qualifying_hands


In [ ]:
def simulate_hand(
    test_hand: List[Card],
    turned_card: Card,
    trump_suit: Suit,
    forced_card: Optional[Card],
    simulation_seed: int,
    dealer_id: int = 0,
    test_player_id: Optional[int] = None,
) -> Tuple[bool, int, int]:
    """
    Simulate a single hand with a forced first trick card.

    Parameters
    ----------
    test_hand : List[Card]
        The test player's hand.
    turned_card : Card
        The turned card.
    trump_suit : Suit
        The trump suit.
    forced_card : Optional[Card]
        The card to force on the first trick. If None, uses normal strategy.
    simulation_seed : int
        Random seed for this simulation (for opponent hands).
    dealer_id : int
        ID of the dealer (default: 0).
    test_player_id : Optional[int]
        ID of the test player. If None, calculated as (dealer_id + 1) % 4.

    Returns
    -------
    Tuple[bool, int, int]
        Tuple of (test_player_team_won, tricks_won_team0, tricks_won_team1).
    """
    # Set seed for this simulation
    random.seed(simulation_seed)
    np.random.seed(simulation_seed)

    # Determine test player ID (first to play after dealer)
    if test_player_id is None:
        test_player_id = (dealer_id + 1) % 4

    # Create game with random opponents
    player_config = [
        ("Player0", "random"),
        ("Player1", "random"),
        ("Player2", "random"),
        ("Player3", "random"),
    ]
    # Set test player to use AI profile
    player_config[test_player_id] = ("TestPlayer", "ai")

    game = Game(player_config)
    game.dealer_id = dealer_id

    # Create full deck and remove test hand and turned card
    full_deck = Deck()
    all_cards = set(full_deck.cards)

    # Remove test hand cards and turned card from available cards
    used_cards = set(test_hand) | {turned_card}
    remaining_cards = [card for card in all_cards if card not in used_cards]

    # Shuffle remaining cards for opponent hands
    random.shuffle(remaining_cards)

    # Set test player's hand
    game.players[test_player_id].receive_hand(test_hand.copy())

    # Deal to opponents from remaining cards
    card_idx = 0
    for i in range(4):
        if i != test_player_id:
            opponent_hand = remaining_cards[card_idx : card_idx + 5]
            game.players[i].receive_hand(opponent_hand)
            card_idx += 5

    # Set turned card
    game.turned_card = turned_card

    # Create forced profile for test player
    base_profile = game.players[test_player_id].profile
    forced_profile = ForcedFirstTrickProfile(forced_card, base_profile)
    game.players[test_player_id].profile = forced_profile

    # Play the hand
    try:
        # Select trump (force the known trump suit)
        trump_selector = TrumpSelector(game.players, None)
        selected_trump = trump_selector.select_trump(turned_card, dealer_id)

        if selected_trump is None or selected_trump != trump_suit:
            # Trump selection didn't match - return as loss
            return False, 0, 0

        game.trump_suit = trump_suit

        # Play 5 tricks and track winners
        tricks_won = [0, 0]  # Team 0 and Team 1

        # Reset trick winner for new hand
        if hasattr(game, "_last_trick_winner"):
            delattr(game, "_last_trick_winner")

        for trick_num in range(5):
            game._current_trick_number = trick_num
            winner_id = game._play_trick()
            winner = game.players[winner_id]
            tricks_won[winner.team] += 1

        # Test player's team
        test_player_team = test_player_id % 2
        test_player_team_won = tricks_won[test_player_team] >= 3

        return test_player_team_won, tricks_won[0], tricks_won[1]

    except Exception as e:
        # If simulation fails, return as loss
        return False, 0, 0


def _simulate_hand_wrapper(args: Tuple) -> Tuple[bool, int, int]:
    """Wrapper function for multiprocessing."""
    (
        test_hand,
        turned_card,
        trump_suit,
        forced_card,
        simulation_seed,
        dealer_id,
        test_player_id,
    ) = args
    return simulate_hand(
        test_hand, turned_card, trump_suit, forced_card, simulation_seed, dealer_id, test_player_id
    )


In [ ]:
def run_monte_carlo_simulation(
    test_hand: List[Card],
    turned_card: Card,
    trump_suit: Suit,
    strategy_card: Optional[Card],
    num_simulations: int,
    base_seed: int,
    dealer_id: int = 0,
    test_player_id: Optional[int] = None,
    num_workers: Optional[int] = None,
) -> Tuple[float, float, List[int], List[int]]:
    """
    Run Monte Carlo simulation for a strategy.

    Parameters
    ----------
    test_hand : List[Card]
        The test player's hand.
    turned_card : Card
        The turned card.
    trump_suit : Suit
        The trump suit.
    strategy_card : Optional[Card]
        The card to play on first trick for this strategy.
    num_simulations : int
        Number of simulations to run.
    base_seed : int
        Base seed for generating simulation seeds.
    dealer_id : int
        ID of the dealer.
    test_player_id : Optional[int]
        ID of the test player.
    num_workers : Optional[int]
        Number of parallel workers. If None, uses CPU count - 1.

    Returns
    -------
    Tuple[float, float, List[int], List[int]]
        Tuple of (win_rate, avg_tricks_won, tricks_won_list, opponent_tricks_list).
    """
    if num_workers is None:
        num_workers = max(1, mp.cpu_count() - 1)

    # Prepare arguments for each simulation
    simulation_args = [
        (
            test_hand,
            turned_card,
            trump_suit,
            strategy_card,
            base_seed + sim_num,
            dealer_id,
            test_player_id,
        )
        for sim_num in range(num_simulations)
    ]

    wins = 0
    tricks_won_list: List[int] = []
    opponent_tricks_list: List[int] = []

    # Run simulations in parallel
    if num_workers > 1 and num_simulations > 10:
        # Use multiprocessing for larger simulations
        with mp.Pool(processes=num_workers) as pool:
            results = list(
                tqdm(
                    pool.imap(_simulate_hand_wrapper, simulation_args),
                    total=num_simulations,
                    desc="Simulations",
                    unit="sim",
                )
            )
    else:
        # Sequential for small simulations or single worker
        results = [
            simulate_hand(
                test_hand,
                turned_card,
                trump_suit,
                strategy_card,
                base_seed + sim_num,
                dealer_id,
                test_player_id,
            )
            for sim_num in tqdm(range(num_simulations), desc="Simulations", unit="sim")
        ]

    # Process results
    for team_won, tricks_team0, tricks_team1 in results:
        if team_won:
            wins += 1
        tricks_won_list.append(tricks_team0)
        opponent_tricks_list.append(tricks_team1)

    win_rate = wins / num_simulations
    avg_tricks_won = statistics.mean(tricks_won_list) if tricks_won_list else 0.0

    return win_rate, avg_tricks_won, tricks_won_list, opponent_tricks_list


## Generate Qualifying Hands


In [ ]:
# Find qualifying hands
qualifying_hands = find_qualifying_hands(
    num_hands=10,  # Start with 10 hands for testing
    min_strength=100.0,
    max_strength=180.0,
    seed_base=42,
)

# Display first few hands
player = HeuristicPlayer()
print(f"\nFirst 3 qualifying hands:")
for i, (hand, turned_card, trump_suit) in enumerate(qualifying_hands[:3]):
    strength = player._evaluate_hand_strength(hand, trump_suit)
    print(f"\nHand {i+1} (Strength: {strength:.1f}, Trump: {trump_suit.value}):")
    for card in hand:
        is_trump = is_card_trump(card, trump_suit)
        print(f"  {card} {'(trump)' if is_trump else '(off-trump)'}")
    print(f"  Turned card: {turned_card}")


## Run Monte Carlo Simulations


In [ ]:
# Configuration
NUM_SIMULATIONS_PER_HAND = 1000  # Start with 1000 per hand
NUM_WORKERS = max(1, mp.cpu_count() - 1)
DEALER_ID = 0
TEST_PLAYER_ID = (DEALER_ID + 1) % 4

print(f"Configuration:")
print(f"  Simulations per hand: {NUM_SIMULATIONS_PER_HAND:,}")
print(f"  Number of workers: {NUM_WORKERS}")
print(f"  Dealer ID: {DEALER_ID}")
print(f"  Test player ID: {TEST_PLAYER_ID} (first to play after dealer)")
print()

# Store results for all hands
all_results = []

# Process each qualifying hand
for hand_idx, (hand, turned_card, trump_suit) in enumerate(qualifying_hands):
    print(f"\n{'='*80}")
    print(f"Hand {hand_idx + 1}/{len(qualifying_hands)}")
    print(f"{'='*80}")
    
    # Find the strategy cards
    trump_nine = None
    ace_off_trump = None
    
    for card in hand:
        if is_card_trump(card, trump_suit) and card.rank == Rank.NINE:
            trump_nine = card
        elif not is_card_trump(card, trump_suit) and card.rank == Rank.ACE:
            ace_off_trump = card
    
    if trump_nine is None or ace_off_trump is None:
        print(f"  Skipping hand - missing required cards")
        continue
    
    print(f"  Strategy 1 card: {trump_nine} (9 trump)")
    print(f"  Strategy 2 card: {ace_off_trump} (Ace off-trump)")
    
    # Calculate hand strength
    strength = player._evaluate_hand_strength(hand, trump_suit)
    print(f"  Hand strength: {strength:.1f}")
    
    # Run simulations for Strategy 1: 9 trump
    print(f"\n  Running Strategy 1: Lead with 9 trump...")
    base_seed = 1000000 * (hand_idx + 1)
    win_rate_1, avg_tricks_1, tricks_list_1, opp_tricks_list_1 = run_monte_carlo_simulation(
        hand, turned_card, trump_suit, trump_nine,
        NUM_SIMULATIONS_PER_HAND, base_seed, DEALER_ID, TEST_PLAYER_ID, NUM_WORKERS
    )
    
    # Run simulations for Strategy 2: Ace off-trump
    print(f"\n  Running Strategy 2: Lead with Ace off-trump...")
    win_rate_2, avg_tricks_2, tricks_list_2, opp_tricks_list_2 = run_monte_carlo_simulation(
        hand, turned_card, trump_suit, ace_off_trump,
        NUM_SIMULATIONS_PER_HAND, base_seed + 1, DEALER_ID, TEST_PLAYER_ID, NUM_WORKERS
    )
    
    # Store results
    all_results.append({
        'hand_idx': hand_idx,
        'hand': hand,
        'trump_suit': trump_suit,
        'hand_strength': strength,
        'strategy1_win_rate': win_rate_1,
        'strategy1_avg_tricks': avg_tricks_1,
        'strategy1_tricks_list': tricks_list_1,
        'strategy2_win_rate': win_rate_2,
        'strategy2_avg_tricks': avg_tricks_2,
        'strategy2_tricks_list': tricks_list_2,
    })
    
    print(f"\n  Results:")
    print(f"    Strategy 1 (9 trump): Win rate = {win_rate_1:.2%}, Avg tricks = {avg_tricks_1:.2f}")
    print(f"    Strategy 2 (Ace off): Win rate = {win_rate_2:.2%}, Avg tricks = {avg_tricks_2:.2f}")
    print(f"    Difference: {win_rate_1 - win_rate_2:+.2%} win rate, {avg_tricks_1 - avg_tricks_2:+.2f} tricks")


## Analyze Results


In [ ]:
# Aggregate results
if len(all_results) > 0:
    df_results = pd.DataFrame([
        {
            'hand_idx': r['hand_idx'],
            'hand_strength': r['hand_strength'],
            'strategy1_win_rate': r['strategy1_win_rate'],
            'strategy1_avg_tricks': r['strategy1_avg_tricks'],
            'strategy2_win_rate': r['strategy2_win_rate'],
            'strategy2_avg_tricks': r['strategy2_avg_tricks'],
            'win_rate_diff': r['strategy1_win_rate'] - r['strategy2_win_rate'],
            'tricks_diff': r['strategy1_avg_tricks'] - r['strategy2_avg_tricks'],
        }
        for r in all_results
    ])
    
    print("="*80)
    print("AGGREGATE RESULTS")
    print("="*80)
    print(f"\nNumber of hands analyzed: {len(all_results)}")
    print(f"\nStrategy 1 (9 trump lead):")
    print(f"  Average win rate: {df_results['strategy1_win_rate'].mean():.2%}")
    print(f"  Average tricks won: {df_results['strategy1_avg_tricks'].mean():.2f}")
    print(f"\nStrategy 2 (Ace off-trump lead):")
    print(f"  Average win rate: {df_results['strategy2_win_rate'].mean():.2%}")
    print(f"  Average tricks won: {df_results['strategy2_avg_tricks'].mean():.2f}")
    print(f"\nOverall difference:")
    print(f"  Win rate difference: {df_results['win_rate_diff'].mean():+.2%}")
    print(f"  Tricks difference: {df_results['tricks_diff'].mean():+.2f}")
    
    # Statistical test
    try:
        from scipy import stats
        
        # Combine all trick lists for t-test
        all_tricks_1 = []
        all_tricks_2 = []
        for r in all_results:
            all_tricks_1.extend(r['strategy1_tricks_list'])
            all_tricks_2.extend(r['strategy2_tricks_list'])
        
        t_stat, p_value = stats.ttest_ind(all_tricks_1, all_tricks_2)
        print(f"\nStatistical Test (t-test):")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_value:.4f}")
        if p_value < 0.05:
            print(f"  Result: Statistically significant difference (p < 0.05)")
        else:
            print(f"  Result: No statistically significant difference (p >= 0.05)")
    except ImportError:
        print("\n(scipy not available for statistical test)")
    
    print("\n" + "="*80)


## Visualizations


if len(all_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Win rate comparison
    ax = axes[0, 0]
    strategies = ['9 Trump\nLead', 'Ace Off-Trump\nLead']
    win_rates = [
        df_results['strategy1_win_rate'].mean(),
        df_results['strategy2_win_rate'].mean(),
    ]
    bars = ax.bar(strategies, win_rates, color=['#2ecc71', '#3498db'], alpha=0.7)
    ax.set_ylabel('Win Rate')
    ax.set_title('Average Win Rate Comparison')
    ax.set_ylim([0, 1])
    for i, (bar, rate) in enumerate(zip(bars, win_rates)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{rate:.2%}', ha='center', va='bottom', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Tricks comparison
    ax = axes[0, 1]
    avg_tricks = [
        df_results['strategy1_avg_tricks'].mean(),
        df_results['strategy2_avg_tricks'].mean(),
    ]
    bars = ax.bar(strategies, avg_tricks, color=['#2ecc71', '#3498db'], alpha=0.7)
    ax.set_ylabel('Average Tricks Won')
    ax.set_title('Average Tricks Won Comparison')
    ax.set_ylim([0, 5])
    for i, (bar, tricks) in enumerate(zip(bars, avg_tricks)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{tricks:.2f}', ha='center', va='bottom', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Win rate difference by hand
    ax = axes[1, 0]
    ax.plot(df_results['hand_idx'], df_results['win_rate_diff'], 'o-', color='#e74c3c', linewidth=2, markersize=8)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Hand Index')
    ax.set_ylabel('Win Rate Difference\n(Strategy 1 - Strategy 2)')
    ax.set_title('Win Rate Difference by Hand')
    ax.grid(alpha=0.3)
    
    # Tricks distribution (combined across all hands)
    ax = axes[1, 1]
    all_tricks_1 = []
    all_tricks_2 = []
    for r in all_results:
        all_tricks_1.extend(r['strategy1_tricks_list'])
        all_tricks_2.extend(r['strategy2_tricks_list'])
    
    ax.hist(all_tricks_1, bins=range(7), alpha=0.6, label='9 Trump Lead', color='#2ecc71', edgecolor='black')
    ax.hist(all_tricks_2, bins=range(7), alpha=0.6, label='Ace Off-Trump Lead', color='#3498db', edgecolor='black')
    ax.set_xlabel('Tricks Won')
    ax.set_ylabel('Frequency')
    ax.set_title('Tricks Won Distribution')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nVisualizations displayed above.")
